# Sentinela Regional Binary Temporal GOES Colab

Run this notebook top-to-bottom in Google Colab from the Sentinela-ModelS repo stored in Google Drive.

Contract:

```text
Input sample: [T, C, H, W] = [4, 9, 64, 64]
Batch input:  [B, T, C, H, W]
Model input:  [B, T*C, H, W] = [B, 36, 64, 64]
Output:       scene_logits [B]
Labels:       0 no_fire, 1 fire_signal
```

This notebook is Drive-only: no GitHub clone step. Pipeline cells use explicit `!python3` commands so Colab streams command output directly.

In [ ]:
#@title 1. Runtime parameters
from pathlib import Path
import os
import sys
import json
import getpass

# Drive repo path. This folder must contain scripts/, src/, configs/, and requirements.txt.
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Sentinela-ModelS") #@param {type:"string"}

REGION = "south_america" #@param ["south_america", "central_america", "caribbean", "north_america"]

PERSIST_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_DATA_ROOT = "/content/drive/MyDrive/sentinela_data/goes_fire" #@param {type:"string"}
LOCAL_DATA_ROOT = "/content/sentinela_data/goes_fire" #@param {type:"string"}

# FIRMS key is used only for bootstrap labels.
# Leave empty to enter it securely when prompted.
FIRMS_MAP_KEY = "" #@param {type:"string"}

MIN_CONFIDENCE = "nominal" #@param ["", "low", "nominal", "high"] {allow-input: true}
START_DATE = "2024-01-01" #@param {type:"string"}
END_DATE = "2026-05-11" #@param {type:"string"}
MAX_FIRMS_ROWS = 2000 #@param {type:"integer"}

TEMPORAL_OFFSETS = "-30,-20,-10,0" #@param {type:"string"}
SAMPLES_PER_CLASS = 1000 #@param {type:"integer"}

# Use 5-10 for a smoke run. Use 0 for the full selected target set.
MAX_GOES_FILES = 8 #@param {type:"integer"}
MAX_TARGETS_PER_FILE = 0 #@param {type:"integer"}
HARD_NEGATIVE_RATIO = 0.5 #@param {type:"number"}

VARIANT = "m" #@param ["n", "s", "m", "l"]
EPOCHS = 20 #@param {type:"integer"}
BATCH_SIZE = 16 #@param {type:"integer"}
NUM_WORKERS = 2 #@param {type:"integer"}
DROP_UNCERTAIN = False #@param {type:"boolean"}

DATA_ROOT = DRIVE_DATA_ROOT if PERSIST_TO_DRIVE else LOCAL_DATA_ROOT
DATA_ROOT_PATH = Path(DATA_ROOT)

PROJECT_ROOT_STR = str(PROJECT_ROOT)
DATA_ROOT_STR = str(DATA_ROOT_PATH)
START_DATE_ARG = f"--start-date {START_DATE}" if START_DATE.strip() else ""
END_DATE_ARG = f"--end-date {END_DATE}" if END_DATE.strip() else ""
MAX_GOES_FILES_ARG = f"--max-goes-files {MAX_GOES_FILES}" if int(MAX_GOES_FILES) > 0 else ""
MAX_TARGETS_PER_FILE_ARG = f"--max-targets-per-file {MAX_TARGETS_PER_FILE}" if int(MAX_TARGETS_PER_FILE) > 0 else ""
DROP_UNCERTAIN_ARG = "--drop-uncertain" if DROP_UNCERTAIN else ""

print(json.dumps({
    "project_root": PROJECT_ROOT_STR,
    "region": REGION,
    "data_root": DATA_ROOT_STR,
    "temporal_offsets": TEMPORAL_OFFSETS,
    "samples_per_class": SAMPLES_PER_CLASS,
    "max_goes_files": MAX_GOES_FILES,
    "epochs": EPOCHS,
    "variant": VARIANT,
}, indent=2))

In [ ]:
#@title 2. Mount Google Drive and verify repo path
from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT_PATH.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT does not exist: {PROJECT_ROOT}\n"
        "Set PROJECT_ROOT to the Drive folder that contains Sentinela-ModelS."
    )

required = ["scripts", "src", "configs", "requirements.txt"]
missing = [name for name in required if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f"PROJECT_ROOT is not a Sentinela-ModelS repo folder. Missing: {missing}")

print(f"Using Drive repo at: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT_PATH}")

In [ ]:
#@title 3. Prepare repo and install dependencies
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
!python3 -m pip install -r requirements.txt

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Dependencies installed.")

In [ ]:
#@title 4. Environment and quick contract check
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['SENTINELA_DATA_ROOT'] = DATA_ROOT_STR

if not FIRMS_MAP_KEY:
    FIRMS_MAP_KEY = getpass.getpass('NASA FIRMS map key: ')
os.environ['FIRMS_MAP_KEY'] = FIRMS_MAP_KEY

import torch
from sentinela_models import SentinelaConfig, SentinelaModel
from sentinela_models.regional import get_region_config

cfg = get_region_config(REGION)
temporal_steps = len([v for v in TEMPORAL_OFFSETS.split(',') if v.strip()])
channels = len(cfg.channels(include_derived=False))
model = SentinelaModel(SentinelaConfig(
    in_channels=channels * temporal_steps,
    mask_classes=1,
    scene_classes=1,
    temporal_steps=temporal_steps,
    input_channels_per_timestep=channels,
    variant=VARIANT,
    input_size=cfg.patch_size,
))
shape_report = model.infer_shapes('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
print('GOES channels:', channels)
print('Temporal steps:', temporal_steps)
print('Infer shapes:', shape_report)

## Data Prep

The next two cells create bootstrap labels from NASA FIRMS and normalize them into the regional label format. FIRMS is a bootstrap source here, not an independent lead-time benchmark.

In [ ]:
#@title 5. Download FIRMS bootstrap labels
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
!python3 scripts/download_firms_bootstrap_labels.py --region "{REGION}" --data-root "{DATA_ROOT_STR}" --min-confidence "{MIN_CONFIDENCE}" --max-rows {MAX_FIRMS_ROWS} {START_DATE_ARG} {END_DATE_ARG}

In [ ]:
#@title 6. Ingest regional labels
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
LABELS_PATH = str(DATA_ROOT_PATH / REGION / 'raw' / 'labels' / 'firms_bootstrap_events.csv')
!python3 scripts/ingest_regional_labels.py --region "{REGION}" --data-root "{DATA_ROOT_STR}" --labels "{LABELS_PATH}"

## Build Temporal GOES Samples

This uses the streaming builder. It downloads the GOES files needed for each temporal sequence, extracts patches, writes `.npz` samples, then deletes raw NetCDF files unless you change the script options.

Default smoke run creates `[4, 9, 64, 64]` samples from offsets `[-30, -20, -10, 0]`.

In [ ]:
#@title 7. Build temporal GOES samples
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
!python3 scripts/build_regional_goes_samples_streaming.py --region "{REGION}" --data-root "{DATA_ROOT_STR}" --samples-per-class {SAMPLES_PER_CLASS} --hard-negative-ratio {HARD_NEGATIVE_RATIO} --temporal-offsets={TEMPORAL_OFFSETS} {START_DATE_ARG} {END_DATE_ARG} {MAX_GOES_FILES_ARG} {MAX_TARGETS_PER_FILE_ARG}

In [ ]:
#@title 8. Inspect manifest and one sample
import csv
import collections
import numpy as np

manifest = DATA_ROOT_PATH / REGION / 'manifest.csv'
if not manifest.exists():
    raise FileNotFoundError(f'Missing manifest: {manifest}')

with manifest.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

label_counts = collections.Counter(row.get('label_name', '') for row in rows)
split_counts = collections.Counter(row.get('split', '') for row in rows)
print('Manifest:', manifest)
print('Rows:', len(rows))
print('By label:', dict(label_counts))
print('By split:', dict(split_counts))

first = DATA_ROOT_PATH / REGION / rows[0]['path']
with np.load(first, allow_pickle=False) as sample:
    print('First sample:', first)
    print('x shape:', sample['x'].shape)
    print('class_label:', int(sample['class_label']))
    if 'temporal_offsets_minutes' in sample:
        print('temporal_offsets_minutes:', sample['temporal_offsets_minutes'].tolist())

## Train and Evaluate

Training uses binary BCE:

```text
0 no_fire = negative + hard_negative
1 fire_signal = active_fire + early_fire_signal
```

`uncertain` is zero-weight by default. Enable `DROP_UNCERTAIN` to remove uncertain rows entirely.

In [ ]:
#@title 9. Train binary temporal model
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
!python3 scripts/train_regional_goes.py --region "{REGION}" --data-root "{DATA_ROOT_STR}" --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --variant "{VARIANT}" --temporal-offsets={TEMPORAL_OFFSETS} {DROP_UNCERTAIN_ARG}

In [ ]:
#@title 10. Evaluate best checkpoint
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
CHECKPOINT = str(PROJECT_ROOT / 'models' / 'regional' / REGION / 'best.pt')
OUT_JSON = str(DATA_ROOT_PATH / REGION / 'evaluation' / 'best_test.json')
!python3 scripts/evaluate_regional_goes.py --region "{REGION}" --data-root "{DATA_ROOT_STR}" --checkpoint "{CHECKPOINT}" --split test --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --out-json "{OUT_JSON}"

In [ ]:
#@title 11. Write report and show final metrics
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
REPORT_EVAL = str(DATA_ROOT_PATH / REGION / 'evaluation' / 'best_test.json')
HISTORY_JSON = str(PROJECT_ROOT / 'models' / 'regional' / REGION / 'history.json')
!python3 scripts/write_regional_benchmark_report.py --region "{REGION}" --data-root "{DATA_ROOT_STR}" --eval-json "{REPORT_EVAL}" --history-json "{HISTORY_JSON}"

result = json.loads(Path(REPORT_EVAL).read_text())
print(json.dumps({
    'accuracy': result.get('accuracy'),
    'precision': result.get('precision'),
    'recall': result.get('recall'),
    'f1': result.get('f1'),
    'labels': result.get('labels'),
    'temporal_offsets_minutes': result.get('temporal_offsets_minutes'),
    'original_class_counts': result.get('original_class_counts'),
}, indent=2))

In [ ]:
#@title 12. Save key artifacts to Drive or list local outputs
artifacts = {
    'best_checkpoint': str(PROJECT_ROOT / 'models' / 'regional' / REGION / 'best.pt'),
    'latest_checkpoint': str(PROJECT_ROOT / 'models' / 'regional' / REGION / 'latest.pt'),
    'history': str(PROJECT_ROOT / 'models' / 'regional' / REGION / 'history.json'),
    'evaluation': str(DATA_ROOT_PATH / REGION / 'evaluation' / 'best_test.json'),
    'report': str(DATA_ROOT_PATH / REGION / 'reports' / 'regional_benchmark.md'),
    'manifest': str(DATA_ROOT_PATH / REGION / 'manifest.csv'),
}
print(json.dumps(artifacts, indent=2))

if PERSIST_TO_DRIVE:
    print('Artifacts are already under Drive data root where applicable.')
else:
    print('Colab runtime storage is ephemeral. Download checkpoints/reports or rerun with PERSIST_TO_DRIVE=True.')

## Real Run Suggested Settings

After the smoke run works, rerun from the top with settings closer to:

```text
MAX_FIRMS_ROWS = 0
SAMPLES_PER_CLASS = 10000
MAX_GOES_FILES = 0
EPOCHS = 20
VARIANT = "s" or "m"
PERSIST_TO_DRIVE = True
```

If GOES availability is sparse, try `TEMPORAL_OFFSETS = "-60,-40,-20,0"` for a wider sequence.